# ARC_ATLAS v4 Slice-Block Small-Lesion Training

This runbook replaces the old patch/window experiment with a 2.5D full-slice design:
- each training step is one entire brain
- the model input is every overlapping 3-slice slab for that brain: `(num_slices, full_x, full_y, 3)`
- the model predicts the center slice for each slab
- validation stitches center-slice probabilities back into a full 3D probability map
- saved validation outputs include both probability NIfTI files and thresholded segmentation NIfTI files

Default geometry is the canonical center-cropped `192 x 224 x 192` brain, sliding along axis `z` with stride `1`.


In [ ]:
from pathlib import Path
import sys
import time

PROJECT_ROOT = Path.cwd()
FALLBACK_PROJECT_ROOT = Path("/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4")
if not (PROJECT_ROOT / "src" / "training_v2_slice_blocks.py").exists():
    PROJECT_ROOT = FALLBACK_PROJECT_ROOT

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import training_v2_slice_blocks as seg

TRAIN_DIR = PROJECT_ROOT / "data" / "splits" / "90_10_random" / "train"
RUN_DIR = PROJECT_ROOT / "runs" / f"{time.strftime('%Y%m%d_%H%M%S')}_slice_blocks"

cfg = seg.SliceBlockTrainingConfig(
    DATA_DIR=TRAIN_DIR,
    IMAGES_DIR=TRAIN_DIR / "t1",
    MASKS_DIR=TRAIN_DIR / "masks",
    MANIFEST_PATH=TRAIN_DIR / "manifest.csv",
    MODEL_DIR=RUN_DIR / "models",
    CALLBACKS_DIR=RUN_DIR / "callbacks",
    TARGET_SHAPE=(192, 224, 192),
    RESAMPLE_TO_TARGET=False,
    SLICE_AXIS=2,
    BLOCK_DEPTH=3,
    SLICE_STRIDE=1,
    TOTAL_EPOCHS=120,
    INITIAL_LR=2e-4,
    BASE_FILTERS=8,
    UNET_DEPTH=4,
    DECISION_THRESHOLD=0.35,
    SAVE_VAL_PREDICTIONS=True,
    NUM_VAL_PREDICTIONS=3,
    FIT_VERBOSE=2,
)

print(f"Training data: {TRAIN_DIR}")
print(f"Run dir: {RUN_DIR}")
print(f"Input per brain: (num_slices, {cfg.input_shape[0]}, {cfg.input_shape[1]}, {cfg.input_shape[2]})")
print(f"Best checkpoint: {cfg.checkpoint_path}")
print(f"Validation outputs: {cfg.CALLBACKS_DIR / 'predictions'}")


In [ ]:
# Shape sanity check before launching the full run.
cases = seg.load_cases(cfg)
image, mask, _ = seg.load_case_arrays(cases[0], cfg)
x, y = seg.make_slice_blocks(image, mask, cfg)
model = seg.build_slice_block_model(cfg)

print(f"Cases: {len(cases)}")
print(f"Prepared brain: image={image.shape}, mask={mask.shape}")
print(f"One-brain batch: x={x.shape}, y={y.shape}")
params = model.count_params()
print(f"Lesion voxels in sanity case: {int(y.sum())}")
print(f"Model params: {params:,}")
del model
seg.tf.keras.backend.clear_session()


In [ ]:
# Launch training. This writes config, split diagnostics, CSV logs, checkpoints,
# whole-brain validation CSV/JSONL, and best-epoch probability/segmentation NIfTI outputs.
history = seg.train_slice_block_model(cfg)


In [ ]:
# Optional: export probability + threshold segmentation for one case from the best checkpoint.
weights = cfg.checkpoint_path if cfg.checkpoint_path.exists() else cfg.latest_path
if not weights.exists():
    raise FileNotFoundError(f"No trained weights found yet: {weights}")

inference_model = seg.build_slice_block_inference_model(cfg, weights_path=weights)
case = seg.load_cases(cfg)[0]
prob, mask, ref_img = seg.predict_case_probability_map(inference_model, case, cfg)
out_dir = cfg.CALLBACKS_DIR / "manual_prediction"
seg.save_case_outputs(out_dir, case, prob, cfg.DECISION_THRESHOLD, ref_img)

print(f"Saved probability map and threshold segmentation to: {out_dir}")
print(f"Probability range: min={float(prob.min()):.6f}, max={float(prob.max()):.6f}")
print(f"Threshold voxels @ {cfg.DECISION_THRESHOLD:.2f}: {int((prob >= cfg.DECISION_THRESHOLD).sum())}")
